# Luku 7: Keskustelusovellusten rakentaminen
## Microsoft Foundry Models API Aloitusopas

Tämä muistikirja on mukautettu [Azure OpenAI Sample Repository](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst) -kokoelmasta, joka sisältää muistikirjoja, jotka käyttävät [Azure OpenAI](aoai-assignment.ipynb) -palveluita.

> **Huom:** GitHub Models poistuu käytöstä heinäkuun 2026 lopussa. Tämä muistikirja käyttää nyt [Microsoft Foundry Models](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst), joka tarjoaa saman ilmaisen kokeiltavan mallikatalogin ja Azure AI Inference SDK -kokemuksen.


# Yleiskatsaus  
"Suuret kielimallit ovat funktioita, jotka kartoittavat tekstiä tekstiksi. Kun annetaan syötteenä tekstijono, suuri kielimalli yrittää ennustaa seuraavaksi tulevan tekstin"(1). Tämä "quickstart" muistikirja esittelee käyttäjille LLM-konseptien yleiskatsauksen, AML:n käyttöönoton keskeiset pakettivaatimukset, pehmeän johdannon kehotemuotoiluun sekä useita lyhyitä esimerkkejä eri käyttötapauksista. 


## Sisällysluettelo  

[Yleiskatsaus](#overview)  
[Kuinka käyttää OpenAI-palvelua](#how-to-use-openai-service)  
[1. Luo OpenAI-palvelusi](#1.-creating-your-openai-service)  
[2. Asennus](#2.-installation)    
[3. Tunnistetiedot](#3.-credentials)  

[Käyttötapaukset](#use-cases)    
[1. Tiivistä teksti](#1.-summarize-text)  
[2. Luokittele teksti](#2.-classify-text)  
[3. Luo uusia tuotenimiä](#3.-generate-new-product-names)  
[4. Hienosäädä luokittelija](#4.fine-tune-a-classifier)  

[Lähteet](#references)


### Rakenna ensimmäinen kysely  
Tämä lyhyt harjoitus tarjoaa perusjohdatuksen mallille annettavien pyyntöjen tekemiseen Microsoft Foundry Models -ympäristössä yksinkertaista "tiivistelmää" varten.


**Vaiheet**:  
1. Asenna `azure-ai-inference`-kirjasto Python-ympäristöösi, jos et ole vielä tehnyt niin.  
2. Lataa vakioapu-kirjastot ja määritä Microsoft Foundry Models -tunnistetietosi.  
3. Valitse malli tehtävääsi varten  
4. Luo yksinkertainen pyyntö mallille  
5. Lähetä pyyntö mallin API:lle!  


### 1. Asenna `azure-ai-inference`


In [ ]:
%pip install azure-ai-inference

### 2. Tuo apukirjastot ja luo käyttöoikeustiedot


In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


### 3. Oikean mallin löytäminen  
GPT-4o- ja GPT-4o mini -mallit voivat ymmärtää ja tuottaa luonnollista kieltä, ja ne ovat saatavilla Microsoft Foundry Models -katalogissa yhdessä Metan, Mistralin, Cohere:n ja Microsoftin mallien kanssa.


In [ ]:
# Select a general purpose chat model
model_name = "gpt-5-mini"


## 4. Kehote suunnittelu  

"Suurten kielimallien taika on siinä, että koulutuksen aikana minimoimalla tämä ennusteen virhe valtavissa tekstimäärissä, mallit oppivat käsitteitä, jotka ovat hyödyllisiä näihin ennusteisiin. Esimerkiksi ne oppivat käsitteitä kuten"(1):

* miten tavata
* miten kielioppi toimii
* miten parafraasoida
* miten vastata kysymyksiin
* miten käydä keskustelua
* miten kirjoittaa monilla kielillä
* miten koodata
* jne.

#### Miten hallita suurta kielimallia  
"Kaikista syötteistä suurelle kielimallille vaikutusvaltaisin on ehdottomasti teksti-kehote(1).

Suuria kielimalleja voidaan ohjata tuottamaan vastauksia muutamalla eri tavalla:

Ohjeistus: Kerro mallille, mitä haluat
Täydennys: Saada malli täydentämään haluamasi alun
Demonstrointi: Näytä mallille mitä haluat, joko:
Muutama esimerkki kehotteessa
Satoja tai tuhansia esimerkkejä hienosäätöaineistossa"



#### Kehotteiden laatimisessa on kolme perussääntöä:

**Näytä ja kerro**. Tee selväksi mitä haluat joko ohjeilla, esimerkeillä tai molempien yhdistelmällä. Jos haluat mallin järjestävän listan aakkosjärjestykseen tai luokittelemaan kappaleen mielialan mukaan, näytä sille, että näin haluat.

**Tarjoa laadukasta dataa**. Jos yrität rakentaa luokittelijaa tai saada mallin seuraamaan kaavaa, varmista että esimerkkejä on tarpeeksi. Oikolue esimerkkisi — malli on yleensä tarpeeksi älykäs huomaamaan perus kirjoitusvirheet ja antaa vastauksen, mutta se voi myös olettaa niiden olevan tahallisia, mikä voi vaikuttaa vastaukseen.

**Tarkista asetuksesi.** Lämpötila ja top_p -asetukset ohjaavat mallin determinismiä vastauksessa. Jos pyydät vastausta, johon on vain yksi oikea vastaus, haluat asettaa nämä matalammiksi. Jos haluat monipuolisempia vastauksia, voit asettaa ne korkeammiksi. Yleisin virhe näiden asetusten kanssa on se, että ihmiset luulevat niiden olevan "älykkyys" tai "luovuus" -ohjaimia.


Lähde: https://learn.microsoft.com/azure/ai-foundry/openai/overview


### 5. Lähetä!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

### Toista sama kutsu, kuinka tulokset vertautuvat?


In [ ]:

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

## Tiivistä teksti  
#### Haaste  
Tiivistä teksti lisäämällä 'tl;dr:' tekstikappaleen loppuun. Huomaa, miten malli ymmärtää suorittaa useita tehtäviä ilman lisäohjeita. Voit kokeilla kuvailevampia kehotteita kuin tl;dr muokataksesi mallin käyttäytymistä ja räätälöidäksesi saamaasi tiivistelmää(3).  

Viimeaikaiset tutkimukset ovat osoittaneet merkittäviä parannuksia monissa NLP-tehtävissä ja vertailuarvoissa esikouluttamalla mallin laajalla tekstikorpuksella ja hienosäätämällä sen jälkeen tiettyä tehtävää varten. Vaikka arkkitehtuuri on tyypillisesti tehtäväagnostinen, tämä menetelmä vaatii edelleen tehtäväkohtaisia hienosäätöaineistoja, joissa on tuhansia tai kymmeniä tuhansia esimerkkejä. Ihmiset sen sijaan pystyvät yleisesti suorittamaan uuden kielitehtävän vain muutamalla esimerkillä tai yksinkertaisilla ohjeilla – jotain, mikä on nykyisille NLP-järjestelmille edelleen haastavaa. Tässä osoitamme, että kielimallien skaalauttaminen parantaa merkittävästi tehtäväagnostista, vähän esimerkkien varassa suoritettavaa toimintaa, joskus jopa saavuttaen kilpailukykyisiä tuloksia aiempiin huipputason hienosäätömenetelmiin verrattuna.  



Tiiv;dr  


# Harjoituksia useisiin käyttötapauksiin  
1. Tiivistä teksti  
2. Luokittele teksti  
3. Luo uusia tuotenimiä


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## Luokittele teksti  
#### Haaste  
Luokittele kohteet ennusteajon aikana annettuihin kategorioihin. Seuraavassa esimerkissä annamme sekä kategoriat että luokiteltavan tekstin kehotteessa (*playground_reference).  

Asiakaskysely: Hei, yksi kannettavan tietokoneeni näppäimistön näppäimistä meni äskettäin rikki ja tarvitsen korvaavan:  

Luokiteltu kategoria:  


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## Luo uusia tuotenimiä
#### Haaste
Luo tuotenimiä esimerkkisanojen perusteella. Tässä annamme kehotteessa tietoa tuotteesta, jolle aiomme luoda nimiä. Tarjoamme myös vastaavan esimerkin osoittaaksemme halutun kaavan. Olemme myös asettaneet lämpötila-arvon korkealle lisätäksemme satunnaisuutta ja innovatiivisempia vastauksia.

Tuotteen kuvaus: Kotikäyttöinen milkshake-kone
Alkusanat: nopea, terveellinen, kompakti.
Tuotenimet: HomeShaker, Fit Shaker, QuickShake, Shake Maker

Tuotteen kuvaus: Kengät, jotka sopivat kaikille jalkakokoille.
Alkusanat: mukautuva, sopiva, omni-fit.


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}])

response.choices[0].message.content

# Viitteet  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry portal](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [Parhaat käytännöt GPT-mallien hienosäätöön tekstin luokittelussa](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# Lisäapua varten  
[OpenAI:n kaupallistamistiimi](AzureOpenAITeam@microsoft.com) 


# Avustajat
* [Chew-Yean Yam](https://www.linkedin.com/in/cyyam/)


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vastuuvapauslauseke**:
Tämä asiakirja on käännetty käyttämällä tekoälypohjaista käännöspalvelua [Co-op Translator](https://github.com/Azure/co-op-translator). Vaikka pyrimme tarkkuuteen, otathan huomioon, että automaattiset käännökset saattavat sisältää virheitä tai epätarkkuuksia. Alkuperäinen asiakirja sen alkuperäiskielellä on virallinen lähde. Tärkeissä asioissa suositellaan ammattimaista ihmiskäännöstä. Emme ole vastuussa tämän käännöksen käytöstä aiheutuvista väärinymmärryksistä tai tulkinnoista.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
